In [157]:
from dotenv import load_dotenv
load_dotenv()

from openai import OpenAI
openai_client = OpenAI()

In [158]:
def rag(self, query):
    search_results = self.search(query)
    prompt = self.build_prompt(query, search_results)
    answer = self.llm(prompt)
    return answer

In [159]:
messages = [
    {"role": "user", "content": "I just discovered the course. Can I join it?"}
]

response = openai_client.responses.create(
    model="gpt-5.4-mini",
    input=messages,
)

response.output_text

'Yes—possibly. If the course is still open for enrollment, you should be able to join it.\n\nA few things to check:\n- whether registration is still open\n- if there are prerequisites\n- whether the course has a waitlist or limited seats\n- whether you need approval from the instructor or registrar\n\nIf you want, I can help you draft a short message asking to join the course.'

In [160]:
def build_context(search_results):
    lines = []

    for doc in search_results:
        lines.append(doc["section"])
        lines.append("Q: " + doc["question"])
        lines.append("A: " + doc["answer"])
        lines.append("")

    return "\n".join(lines).strip()

In [161]:
from ingest import load_faq_data, build_index

documents = load_faq_data()
index = build_index(documents)

In [162]:
from rag_helper import RAGBase

instructions = """
You're a course teaching assistant.
Answer the QUESTION based on the CONTEXT from the FAQ database.
Use only the facts from the CONTEXT when answering the QUESTION.
""".strip()

assistant = RAGBase(
    index=index,
    llm_client=openai_client,
    instructions=instructions,
)

In [163]:
def search(query):
    boost_dict = {'question': 3.0, 'section': 0.5}
    filter_dict = {'course': 'llm-zoomcamp'}

    return index.search(
        query,
        num_results=5,
        boost_dict=boost_dict,
        filter_dict=filter_dict
    )

In [164]:
search_tool = {
    "type": "function",
    'name': 'search',
    'description': 'Search the FAQ database for entries matching the given query.',
    'parameters': {
        "type": "object",
        "properties": {
            'query': {
                "type": "string",
                'description': 'Search query text to look up in the course FAQ.'
            }
        },
        "required": ["query"],
        'additionalProperties': False
    }
}

In [165]:
response = openai_client.responses.create(
    model="gpt-5.4-mini",
    input=messages,
    tools=[search_tool],
)

response.output

[ResponseFunctionToolCall(arguments='{"query":"join course discovered can I join it enrollment registration late join"}', call_id='call_OUPoyWpWskv99IFa9cszPhYU', name='search', type='function_call', id='fc_0e3e1c32bb44334d006a3839b573a881a18315f0ff2ac86b61', namespace=None, status='completed')]

In [166]:
len(response.output)

1

In [167]:
import json

call = response.output[0]
call

ResponseFunctionToolCall(arguments='{"query":"join course discovered can I join it enrollment registration late join"}', call_id='call_OUPoyWpWskv99IFa9cszPhYU', name='search', type='function_call', id='fc_0e3e1c32bb44334d006a3839b573a881a18315f0ff2ac86b61', namespace=None, status='completed')

In [168]:
import json
args = json.loads(call.arguments)
args


{'query': 'join course discovered can I join it enrollment registration late join'}

In [169]:
call.name

'search'

In [170]:
results = search(**args)

In [171]:
result_json = json.dumps(results, indent=2)
result_json

'[\n  {\n    "course": "llm-zoomcamp",\n    "section": "General Course-Related Questions",\n    "question": "I just discovered the course. Can I still join?",\n    "answer": "Yes, but if you want to receive a certificate, you need to submit your project while we\\u2019re still accepting submissions.",\n    "doc_id": "74eb249bbf"\n  },\n  {\n    "course": "llm-zoomcamp",\n    "section": "General Course-Related Questions",\n    "question": "Course: I have registered for the LLM Zoomcamp. When can I expect to receive the confirmation email?",\n    "answer": "You don\'t need it. You\'re accepted. You can also just start learning and submitting homework (while the form is open) without registering. It is not checked against any registered list. Registration is just to gauge interest before the start date.",\n    "doc_id": "977bf7786c"\n  },\n  {\n    "course": "llm-zoomcamp",\n    "section": "General Course-Related Questions",\n    "question": "Certificate: Can I follow the course in a self

In [172]:
function_call_output = {
    "type": "function_call_output",
    'call_id': call.call_id,
    'output': result_json,
}

In [173]:
messages.append(call)

In [174]:
messages.append(function_call_output)

In [175]:
messages

[{'role': 'user', 'content': 'I just discovered the course. Can I join it?'},
 ResponseFunctionToolCall(arguments='{"query":"join course discovered can I join it enrollment registration late join"}', call_id='call_OUPoyWpWskv99IFa9cszPhYU', name='search', type='function_call', id='fc_0e3e1c32bb44334d006a3839b573a881a18315f0ff2ac86b61', namespace=None, status='completed'),
 {'type': 'function_call_output',
  'call_id': 'call_OUPoyWpWskv99IFa9cszPhYU',
  'output': '[\n  {\n    "course": "llm-zoomcamp",\n    "section": "General Course-Related Questions",\n    "question": "I just discovered the course. Can I still join?",\n    "answer": "Yes, but if you want to receive a certificate, you need to submit your project while we\\u2019re still accepting submissions.",\n    "doc_id": "74eb249bbf"\n  },\n  {\n    "course": "llm-zoomcamp",\n    "section": "General Course-Related Questions",\n    "question": "Course: I have registered for the LLM Zoomcamp. When can I expect to receive the confirmat

In [176]:
response = openai_client.responses.create(
    model='gpt-5.4-mini',
    input=messages,
    tools=[search_tool]
)

In [177]:
print(response.output_text)

Yes, you can still join the course.

If you want a certificate, make sure to submit your project while submissions are still open. If you’re just starting to learn, you can begin anytime with the materials.


In [178]:
usage = response.usage
usage.input_tokens, usage.output_tokens

(777, 47)

In [179]:
def calculate_gpt54mini_price(input_tokens, output_tokens):
    # Prices per 1M tokens (example pricing)
    INPUT_PRICE_PER_MILLION = 0.15   # $0.15 / 1M input tokens
    OUTPUT_PRICE_PER_MILLION = 0.60  # $0.60 / 1M output tokens

    input_cost = (input_tokens / 1_000_000) * INPUT_PRICE_PER_MILLION
    output_cost = (output_tokens / 1_000_000) * OUTPUT_PRICE_PER_MILLION

    total_cost = input_cost + output_cost

    return {
        "input_cost": input_cost,
        "output_cost": output_cost,
        "total_cost": total_cost
    }


# Your tokens
result = calculate_gpt54mini_price(652, 33)

print("Total Cost: $", round(result["total_cost"], 8))

Total Cost: $ 0.0001176


In [180]:
def make_call(call):
    args = json.loads(call.arguments)

    if call.name == 'search':
        result = search(**args)

    result_json = json.dumps(result, indent=2)

    return {
        "type": "function_call_output",
        'call_id': call.call_id,
        'output': result_json,
    }

In [181]:
instructions = """
You're a course teaching assistant.
You're given a question from a course student and your task is to answer it.

If you want to look up information, use the search function. 
Use as many keywords from the user question as possible when making first requests.

Make multiple searches.

Try to expand your search by using new keywords
based on the results you get from the search.

At the end, ask if there are other areas that the user wants to explore.
"""

question = 'I just discovered the course. Can I join it?'


messages = [
    {'role': 'developer', 'content': instructions},
    {'role': 'user', 'content': question}
]

In [182]:
response = openai_client.responses.create(
    model='gpt-5.4-mini',
    input=messages,
    tools=[search_tool]
)

In [183]:
response.output

[ResponseFunctionToolCall(arguments='{"query":"join course discovered course can I join enrollment FAQ"}', call_id='call_yyuEWtqvsXEFBcBg3DlUfyXz', name='search', type='function_call', id='fc_0f83b32b2f4bcd82006a3839b80f408191804a639447393102', namespace=None, status='completed'),
 ResponseFunctionToolCall(arguments='{"query":"new student enroll course late join FAQ"}', call_id='call_KwXdWXJ437Quy8kx9DOUnK6Q', name='search', type='function_call', id='fc_0f83b32b2f4bcd82006a3839b80f548191b6677d2ae863f452', namespace=None, status='completed')]

In [184]:
messages.extend(response.output)

for item in response.output:
    if item.type == 'function_call':
        print('function_call:', item.name, item.arguments)
        call_output = make_call(item)
        messages.append(call_output)

    elif item.type == 'message':
        print('ASSISTANT:')
        print(item.content[0].text)

function_call: search {"query":"join course discovered course can I join enrollment FAQ"}
function_call: search {"query":"new student enroll course late join FAQ"}


In [185]:
messages

[{'role': 'developer',
  'content': "\nYou're a course teaching assistant.\nYou're given a question from a course student and your task is to answer it.\n\nIf you want to look up information, use the search function. \nUse as many keywords from the user question as possible when making first requests.\n\nMake multiple searches.\n\nTry to expand your search by using new keywords\nbased on the results you get from the search.\n\nAt the end, ask if there are other areas that the user wants to explore.\n"},
 {'role': 'user', 'content': 'I just discovered the course. Can I join it?'},
 ResponseFunctionToolCall(arguments='{"query":"join course discovered course can I join enrollment FAQ"}', call_id='call_yyuEWtqvsXEFBcBg3DlUfyXz', name='search', type='function_call', id='fc_0f83b32b2f4bcd82006a3839b80f408191804a639447393102', namespace=None, status='completed'),
 ResponseFunctionToolCall(arguments='{"query":"new student enroll course late join FAQ"}', call_id='call_KwXdWXJ437Quy8kx9DOUnK6Q',

In [186]:
messages = [
    {'role': 'developer', 'content': instructions},
    {'role': 'user', 'content': question}
]

it = 1

while True:
    print(f'iteration #{it}...')
    has_function_calls = False

    response = openai_client.responses.create(
        model='gpt-5.4-mini',
        input=messages,
        tools=[search_tool]
    )

    messages.extend(response.output)

    for item in response.output:
        if item.type == 'function_call':
            print('function_call:', item.name, item.arguments)
            call_output = make_call(item)
            messages.append(call_output)
            has_function_calls = True

        elif item.type == 'message':
            print('ASSISTANT:')
            print(item.content[0].text)
    
    it = it + 1
    if has_function_calls == False:
        break

iteration #1...
function_call: search {"query":"join course discovered the course can I join it enrollment signup access"}
function_call: search {"query":"course discovered can I join enrollment late join access FAQ"}
iteration #2...
ASSISTANT:
Yes — you can still join the course.

If you want a certificate, make sure you submit your project while submissions are still open. If you only follow it self-paced after the cohort, you won’t be eligible for a certificate.

If you want, I can also point you to the course docs and the best way to start.


In [187]:
def agent_loop(instructions, question, model='gpt-5.4-mini') -> str:
    messages = [
        {'role': 'developer', 'content': instructions},
        {'role': 'user', 'content': question}
    ]

    it = 1

    while True:
        print(f'iteration #{it}...')
        has_function_calls = False

        response = openai_client.responses.create(
            model=model,
            input=messages,
            tools=[search_tool]
        )

        messages.extend(response.output)

        for item in response.output:
            if item.type == 'function_call':
                print('function_call:', item.name, item.arguments)
                call_output = make_call(item)
                messages.append(call_output)
                has_function_calls = True

            elif item.type == 'message':
                print('ASSISTANT:')
                last_answer = item.content[0].text
                print(item.content[0].text)

        it = it + 1
        if has_function_calls == False:
            break
    
    return last_answer

In [188]:
instructions = """
You're a course teaching assistant.
You're given a question from a course student and your task is to answer it.

If you want to look up information, use the search function. 
Use as many keywords from the user question as possible when making first requests.

Make multiple searches. First perform search, analyze the results 
and then perform more searchers. 

At the end, ask if there are other areas that the user wants to explore.
"""

question = 'I just discovered the course. Can I join it?'

In [189]:
result = agent_loop(instructions, question)

iteration #1...
function_call: search {"query":"join course discovered course can I join enrollment late join discovered the course"}
iteration #2...
function_call: search {"query":"LLM Zoomcamp certificate project submissions while accepting submissions live cohort self-paced peer-review closed form"}
iteration #3...
ASSISTANT:
Yes — you can still join the course and follow along.

If you want a certificate, make sure to submit your project while the course is still accepting submissions, since certificates are only awarded in the live cohort, not in self-paced mode.

If you want, I can also explain how to start the course and what the usual workflow looks like.


In [190]:
result

'Yes — you can still join the course and follow along.\n\nIf you want a certificate, make sure to submit your project while the course is still accepting submissions, since certificates are only awarded in the live cohort, not in self-paced mode.\n\nIf you want, I can also explain how to start the course and what the usual workflow looks like.'

In [191]:
instructions = """
You're a course teaching assistant.
You're given a question from a course student and your task is to answer it.

If you want to look up information, use the search function. 
Use as many keywords from the user question as possible when making first requests.

Make multiple searches. First perform search, analyze the results 
and then perform more searchers. 

The question has to be about the course or its logistics, offtopic questions 
shouldn't be answered. If the search returns nothing, it's likely an off-topic question.
If you can't answer the question using FAQ, don't do it yourself. Only use the 
facts from the FAQ database.

At the end, ask if there are other areas that the user wants to explore.
"""

question = "what's queen gambit?"

result = agent_loop(instructions, question)

iteration #1...
function_call: search {"query":"queen gambit"}
iteration #2...
function_call: search {"query":"chess opening queen's gambit queens gambit"}
iteration #3...
ASSISTANT:
I couldn’t find a course FAQ entry for “queen gambit,” so it looks like this may be off-topic for the course.

If you meant the chess opening, I’d be happy to help in a course-related way only if there’s a relevant FAQ entry. Is there another course topic or logistics question you want to explore?
